# Simload scenario comparison

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from teerex.analysis import (
    read_many_csv,
    infer_column,
    save_summary,
    plot_boxplot,
    plot_completion_throughput,
    TIME_CANDIDATES,
    EVENT_ID_CANDIDATES,
)

## Configure input files

Change `csv_glob` to point to your Simload CSV outputs.

In [ ]:
csv_glob = "../simload_runs/*.csv"
csv_glob = "../simload_runs/out_g4_ray_async.csv"
csv_glob = "../simload_runs/out_g4_ray.csv"

outdir = Path("../simload_analysis")
outdir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(Path(".").glob(csv_glob))
csv_files

## Load and combine CSV files

In [ ]:
df = read_many_csv(csv_files)
df.to_csv(outdir / "combined.csv", index=False)

print(f"rows: {len(df)}")
print(f"runs: {df['run'].nunique()}")
df.head()

In [ ]:
df.columns

## Generate comparison plots

The plot functions save PNG files into `outdir`.

In [ ]:
def plot_timeseries(
    df: pd.DataFrame,
    time_col: str,
    metrics: list[str] | str,
    outdir: Path = None,
) -> None:
    if isinstance(metrics, str):
        metrics = [metrics]

    if not metrics:
        return

    fig, ax = plt.subplots(figsize=(8, 4))

    for run, g in df.groupby("run", dropna=False):
        for metric in metrics:
            if metric not in g.columns:
                continue

            subset = g[[time_col, metric]].dropna().sort_values(time_col)
            if subset.empty:
                continue

            ax.plot(
                subset[time_col],
                subset[metric],
                label=f"{run} | {metric}",
                linewidth=1.5,
            )

    ax.set_xlabel(time_col)
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    if outdir is not None:
        fig.savefig(outdir / "timeseries_multi_metrics.png", dpi=160)
        plt.close(fig)

In [ ]:
plot_timeseries(df, 'start_time', ['cpu_runtime_s', 'gpu_runtime_s'])


In [ ]:
timeline = df[
    [
        "run",
        "event_id",
        "cpu_start_time",
        "cpu_end_time",
        "gpu_start_time",
        "gpu_end_time",
    ]
].dropna().copy()

time_origin = timeline[["cpu_start_time", "gpu_start_time"]].min().min()

for col in ["cpu_start_time", "cpu_end_time", "gpu_start_time", "gpu_end_time"]:
    timeline[f"{col}_offset_s"] = timeline[col] - time_origin

timeline = timeline.sort_values(["run", "event_id"]).reset_index(drop=True)

resource_y = {"GPU": 0, "CPU": 1}

fig, ax = plt.subplots(figsize=(11, 3))

event_starts = timeline["cpu_start_time_offset_s"].dropna().sort_values().unique()
for event_start in event_starts:
    ax.axvline(event_start, color="0.2", linewidth=0.8, alpha=0.5, zorder=0)

labels_drawn = {"CPU": False, "GPU": False}

for _, row in timeline.iterrows():
    cpu_start = row["cpu_start_time_offset_s"]
    cpu_end   = row["cpu_end_time_offset_s"]
    gpu_start = row["gpu_start_time_offset_s"]
    gpu_end   = row["gpu_end_time_offset_s"]

    ax.hlines(
        resource_y["CPU"],
        cpu_start,
        cpu_end,
        color="tab:blue",
        linewidth=10,
        label="CPU busy" if not labels_drawn["CPU"] else "_nolegend_",
    )
    ax.hlines(
        resource_y["GPU"],
        gpu_start,
        gpu_end,
        color="tab:orange",
        linewidth=10,
        label="GPU busy" if not labels_drawn["GPU"] else "_nolegend_",
    )
    labels_drawn["CPU"] = True
    labels_drawn["GPU"] = True

ax.set_xlabel("time since first stage start (s)")
ax.set_ylabel("resource")
ax.set_yticks([resource_y["GPU"], resource_y["CPU"]])
ax.set_yticklabels(["GPU", "CPU"])
ax.set_ylim(-0.6, 1.6)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")

fig.tight_layout()
#fig.savefig(outdir / "cpu_gpu_busy_intervals.png", dpi=160)
#plt.show()


In [ ]:
timeline